# Silver project

## Purpose

Publish one canonical DPWH project per Contract ID. Every value comes from `silver_dpwh_project_component`. A value is published only when the components agree. Conflicts stay visible.

## Grain

One project per Contract ID, per selected source snapshot and Silver run.

- Primary key: `project_key`, a SHA-256 of the source system label and Contract ID. It is the same on every run.
- `component_count` links each project back to its component rows.

## Inputs

- `02-silver.silver_dpwh_project_component`, one run only.

## Outputs

- `02-silver.silver_project`

## Not done here

- No Bronze read. The governed category result is consolidated from component arrays without another config join or JSON parse.
- Budgets are never added across components. Repeated identical budgets resolve to one value. Different budgets stay as a `CONFLICT`.
- Long-running and stalled flags stay `NULL` until the team approves their rules.

## 1. Confirm the component input is safe to group

**What:** check that the component table holds one run and that every row has a Contract ID.

**Why:** grouping rows from two runs would mix snapshots. A blank Contract ID cannot be grouped safely.

**Protects:** Timeliness and Validity. The notebook stops before any write.

**Expected result:** both checks pass.

In [0]:
%sql
USE CATALOG `buildabida-capstone`;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT _source_snapshot_id) = 1,
    'STOP: silver_dpwh_project_component must hold exactly one run and one snapshot'
)
FROM `02-silver`.silver_dpwh_project_component;

SELECT ASSERT_TRUE(
    COUNT_IF(contract_id IS NULL OR TRIM(contract_id) = '') = 0,
    'STOP: project consolidation cannot safely group missing Contract IDs'
)
FROM `02-silver`.silver_dpwh_project_component;

CREATE OR REPLACE TEMPORARY VIEW project_parameters AS
SELECT
    'silver_project_v3' AS transformation_rule_version,
    'dpwh' AS project_key_label;

## 2. Resolve each attribute the same way

**What:** for each attribute, count distinct values, publish the value only when there is exactly one, and say why when there is not.

**Why:**

- One pattern for every attribute keeps the rule easy to check.
- The distinct count and the published value now use the same expression. Before, `reported_region`, `contractor`, and `source_of_funds` counted untrimmed values but published trimmed ones, so a value that differed only by spaces showed a count of 2 and a resolved value.
- `reported_region` now has a resolution status, so Table 13 can tell a missing region from conflicting ones.
- `infra_year_parsed` and its status come from the component parse. Gold no longer needs `TRY_CAST`.
- `MAX` here only reads the one value that exists. It never picks a winner.

**Protects:** Consistency and Accuracy.

**Expected result:** one row per Contract ID.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW project_attribute_resolution AS
SELECT
    contract_id,
    COUNT(*) AS component_count,

    COUNT(DISTINCT NULLIF(TRIM(project_description_raw), '')) AS distinct_project_description_raw_count,
    CASE WHEN COUNT(DISTINCT NULLIF(TRIM(project_description_raw), '')) = 1 THEN MAX(NULLIF(TRIM(project_description_raw), '')) END AS project_description_raw,
    CASE
        WHEN COUNT(DISTINCT NULLIF(TRIM(project_description_raw), '')) = 0 THEN 'MISSING'
        WHEN COUNT(DISTINCT NULLIF(TRIM(project_description_raw), '')) = 1 THEN 'RESOLVED'
        ELSE 'CONFLICT'
    END AS project_description_raw_resolution_status,
    COUNT(DISTINCT NULLIF(TRIM(category_raw), '')) AS distinct_category_raw_count,
    CASE WHEN COUNT(DISTINCT NULLIF(TRIM(category_raw), '')) = 1 THEN MAX(NULLIF(TRIM(category_raw), '')) END AS category_raw,
    CASE
        WHEN COUNT(DISTINCT NULLIF(TRIM(category_raw), '')) = 0 THEN 'MISSING'
        WHEN COUNT(DISTINCT NULLIF(TRIM(category_raw), '')) = 1 THEN 'RESOLVED'
        ELSE 'CONFLICT'
    END AS category_raw_resolution_status,
    COUNT(DISTINCT NULLIF(TRIM(status_raw), '')) AS distinct_status_raw_count,
    CASE WHEN COUNT(DISTINCT NULLIF(TRIM(status_raw), '')) = 1 THEN MAX(NULLIF(TRIM(status_raw), '')) END AS status_raw,
    CASE
        WHEN COUNT(DISTINCT NULLIF(TRIM(status_raw), '')) = 0 THEN 'MISSING'
        WHEN COUNT(DISTINCT NULLIF(TRIM(status_raw), '')) = 1 THEN 'RESOLVED'
        ELSE 'CONFLICT'
    END AS status_raw_resolution_status,
    COUNT(DISTINCT physical_progress_pct) AS distinct_physical_progress_pct_count,
    CASE WHEN COUNT(DISTINCT physical_progress_pct) = 1 THEN MAX(physical_progress_pct) END AS physical_progress_pct,
    CASE
        WHEN COUNT(DISTINCT physical_progress_pct) = 0 THEN 'MISSING_OR_INVALID'
        WHEN COUNT(DISTINCT physical_progress_pct) = 1 THEN 'RESOLVED'
        ELSE 'CONFLICT'
    END AS physical_progress_pct_resolution_status,
    COUNT(DISTINCT start_date_parsed) AS distinct_start_date_count,
    CASE WHEN COUNT(DISTINCT start_date_parsed) = 1 THEN MAX(start_date_parsed) END AS start_date,
    CASE
        WHEN COUNT(DISTINCT start_date_parsed) = 0 THEN 'MISSING_OR_INVALID'
        WHEN COUNT(DISTINCT start_date_parsed) = 1 THEN 'RESOLVED'
        ELSE 'CONFLICT'
    END AS start_date_resolution_status,
    COUNT(DISTINCT completion_date_parsed) AS distinct_completion_date_count,
    CASE WHEN COUNT(DISTINCT completion_date_parsed) = 1 THEN MAX(completion_date_parsed) END AS completion_date,
    CASE
        WHEN COUNT(DISTINCT completion_date_parsed) = 0 THEN 'MISSING_OR_INVALID'
        WHEN COUNT(DISTINCT completion_date_parsed) = 1 THEN 'RESOLVED'
        ELSE 'CONFLICT'
    END AS completion_date_resolution_status,
    COUNT(DISTINCT NULLIF(TRIM(reported_region), '')) AS distinct_reported_region_count,
    CASE WHEN COUNT(DISTINCT NULLIF(TRIM(reported_region), '')) = 1 THEN MAX(NULLIF(TRIM(reported_region), '')) END AS reported_region,
    CASE
        WHEN COUNT(DISTINCT NULLIF(TRIM(reported_region), '')) = 0 THEN 'MISSING'
        WHEN COUNT(DISTINCT NULLIF(TRIM(reported_region), '')) = 1 THEN 'RESOLVED'
        ELSE 'CONFLICT'
    END AS reported_region_resolution_status,
    COUNT(DISTINCT infra_year_parsed) AS distinct_infra_year_parsed_count,
    CASE WHEN COUNT(DISTINCT infra_year_parsed) = 1 THEN MAX(infra_year_parsed) END AS infra_year_parsed,
    CASE
        WHEN COUNT(DISTINCT infra_year_parsed) = 0 THEN 'MISSING_OR_INVALID'
        WHEN COUNT(DISTINCT infra_year_parsed) = 1 THEN 'RESOLVED'
        ELSE 'CONFLICT'
    END AS infra_year_parsed_resolution_status,
    COUNT(DISTINCT NULLIF(TRIM(infra_year), '')) AS distinct_infra_year_count,
    CASE WHEN COUNT(DISTINCT NULLIF(TRIM(infra_year), '')) = 1 THEN MAX(NULLIF(TRIM(infra_year), '')) END AS infra_year,
    COUNT(DISTINCT NULLIF(TRIM(contractor), '')) AS distinct_contractor_count,
    CASE WHEN COUNT(DISTINCT NULLIF(TRIM(contractor), '')) = 1 THEN MAX(NULLIF(TRIM(contractor), '')) END AS contractor,
    COUNT(DISTINCT NULLIF(TRIM(source_of_funds), '')) AS distinct_source_of_funds_count,
    CASE WHEN COUNT(DISTINCT NULLIF(TRIM(source_of_funds), '')) = 1 THEN MAX(NULLIF(TRIM(source_of_funds), '')) END AS source_of_funds,
    COUNT(DISTINCT latitude_parsed) AS distinct_latitude_parsed_count,
    CASE WHEN COUNT(DISTINCT latitude_parsed) = 1 THEN MAX(latitude_parsed) END AS latitude_parsed,
    COUNT(DISTINCT longitude_parsed) AS distinct_longitude_parsed_count,
    CASE WHEN COUNT(DISTINCT longitude_parsed) = 1 THEN MAX(longitude_parsed) END AS longitude_parsed,
    COUNT(DISTINCT NULLIF(TRIM(standardized_status), '')) AS distinct_standardized_status_count,
    CASE WHEN COUNT(DISTINCT NULLIF(TRIM(standardized_status), '')) = 1 THEN MAX(NULLIF(TRIM(standardized_status), '')) END AS standardized_status,
    COUNT(DISTINCT NULLIF(TRIM(status_group), '')) AS distinct_status_group_count,
    CASE WHEN COUNT(DISTINCT NULLIF(TRIM(status_group), '')) = 1 THEN MAX(NULLIF(TRIM(status_group), '')) END AS status_group,
    COUNT(DISTINCT status_mapping_version) AS distinct_status_mapping_version_count,
    CASE WHEN COUNT(DISTINCT status_mapping_version) = 1 THEN MAX(status_mapping_version) END AS status_mapping_version,
    COUNT(DISTINCT NULLIF(TRIM(status_mapping_state), '')) AS distinct_status_mapping_state_count,
    CASE WHEN COUNT(DISTINCT NULLIF(TRIM(status_mapping_state), '')) = 1 THEN MAX(NULLIF(TRIM(status_mapping_state), '')) END AS status_mapping_state,
    COUNT(DISTINCT NULLIF(TRIM(source_system), '')) AS distinct_source_system_count,
    CASE WHEN COUNT(DISTINCT NULLIF(TRIM(source_system), '')) = 1 THEN MAX(NULLIF(TRIM(source_system), '')) END AS source_system,
    COUNT(DISTINCT _source_snapshot_id) AS distinct_source_snapshot_id_count,
    CASE WHEN COUNT(DISTINCT _source_snapshot_id) = 1 THEN MAX(_source_snapshot_id) END AS source_snapshot_id,
    COUNT(DISTINCT _ingest_run_id) AS distinct_source_ingest_run_id_count,
    CASE WHEN COUNT(DISTINCT _ingest_run_id) = 1 THEN MAX(_ingest_run_id) END AS source_ingest_run_id,
    COUNT(DISTINCT run_id) AS distinct_component_run_id_count,
    CASE WHEN COUNT(DISTINCT run_id) = 1 THEN MAX(run_id) END AS component_run_id,

    CASE
        WHEN COUNT(DISTINCT physical_progress_pct) = 0 THEN 'MISSING_OR_UNPARSEABLE'
        WHEN COUNT(DISTINCT physical_progress_pct) = 1
            THEN MAX(CASE WHEN physical_progress_pct IS NOT NULL THEN progress_quality_status END)
        ELSE 'CONFLICT'
    END AS progress_quality_status,

    MAX(_source_modified_at) AS source_modified_at
FROM `02-silver`.silver_dpwh_project_component
GROUP BY contract_id;

CREATE OR REPLACE TEMPORARY VIEW project_category_resolution AS
WITH category_sets AS (
    SELECT
        contract_id,
        ARRAY_SORT(ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(source_infra_types)))) AS source_infra_types,
        ARRAY_SORT(ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(approved_standardized_sectors)))) AS approved_standardized_sectors,
        ARRAY_SORT(ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(approved_standardized_categories)))) AS approved_standardized_categories,
        ARRAY_SORT(ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(unapproved_source_infra_types)))) AS unapproved_source_infra_types,
        COUNT_IF(is_dpwh_flood_related) AS flood_component_count,
        COUNT(DISTINCT taxonomy_version) AS taxonomy_version_count,
        MAX(taxonomy_version) AS taxonomy_version
    FROM `02-silver`.silver_dpwh_project_component
    GROUP BY contract_id
)
SELECT
    contract_id,
    source_infra_types,
    NULLIF(ARRAY_JOIN(source_infra_types, ', '), '') AS source_infra_type,
    approved_standardized_sectors,
    approved_standardized_categories,
    unapproved_source_infra_types,
    CASE
        WHEN SIZE(source_infra_types) = 0 OR SIZE(unapproved_source_infra_types) > 0 THEN NULL
        WHEN SIZE(source_infra_types) = 1 THEN ELEMENT_AT(approved_standardized_sectors, 1)
        ELSE 'Multi-sector'
    END AS standardized_sector,
    CASE
        WHEN SIZE(source_infra_types) = 0 OR SIZE(unapproved_source_infra_types) > 0 THEN NULL
        WHEN SIZE(source_infra_types) = 1 THEN ELEMENT_AT(approved_standardized_categories, 1)
        ELSE 'Multi-sector'
    END AS standardized_category,
    taxonomy_version,
    CASE
        WHEN SIZE(source_infra_types) > 0 AND SIZE(unapproved_source_infra_types) = 0 THEN 'MAPPED'
        ELSE 'UNMAPPED'
    END AS category_mapping_state,
    CASE
        WHEN SIZE(source_infra_types) = 0 THEN 'MISSING_SOURCE_CATEGORY'
        WHEN SIZE(unapproved_source_infra_types) > 0 THEN 'UNAPPROVED_SOURCE_CATEGORY'
        WHEN SIZE(source_infra_types) = 1 THEN 'MAPPED_SINGLE'
        ELSE 'MAPPED_MULTI'
    END AS category_classification_status,
    CASE
        WHEN SIZE(source_infra_types) = 0
            OR SIZE(unapproved_source_infra_types) > 0
            THEN NULL
        WHEN flood_component_count > 0 THEN TRUE
        ELSE FALSE
    END AS is_dpwh_flood_related,
    taxonomy_version_count
FROM category_sets;

## 3. Protect budgets from double counting

**What:** resolve one reported budget per project, and carry the budget quality status from the components.

**Why:** repeated Contract ID rows may repeat the same budget. Adding them would inflate investment. Different budgets need an approved business rule, so they stay `CONFLICT` with a `NULL` value. See decision D-04.

**Protects:** Accuracy and Consistency.

**Expected result:** no project has a resolved budget when its components disagree.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW project_budget_resolution AS
SELECT
    contract_id,
    COUNT(DISTINCT reported_budget_pesos) AS distinct_budget_value_count,
    COUNT(reported_budget_pesos) AS valid_budget_component_count,
    CASE
        WHEN COUNT(DISTINCT reported_budget_pesos) = 1 THEN MAX(reported_budget_pesos)
    END AS reported_budget_pesos,
    CASE
        WHEN COUNT(DISTINCT reported_budget_pesos) = 0 THEN 'MISSING_OR_INVALID'
        WHEN COUNT(DISTINCT reported_budget_pesos) = 1 AND COUNT(reported_budget_pesos) = 1
            THEN 'RESOLVED_SINGLE'
        WHEN COUNT(DISTINCT reported_budget_pesos) = 1 THEN 'RESOLVED_REPEATED_IDENTICAL'
        ELSE 'CONFLICT'
    END AS budget_resolution_status,
    CASE
        WHEN COUNT(DISTINCT reported_budget_pesos) = 0 THEN 'MISSING_OR_UNPARSEABLE'
        WHEN COUNT(DISTINCT reported_budget_pesos) = 1
            THEN MAX(CASE WHEN reported_budget_pesos IS NOT NULL THEN budget_quality_status END)
        ELSE 'CONFLICT'
    END AS budget_quality_status
FROM `02-silver`.silver_dpwh_project_component
GROUP BY contract_id;

## 4. Publish one project row

**What:** join the attribute and budget results and replace the table in one write.

**Why:**

- Column names that downstream tables read stay the same. New columns are added, none are removed.
- `zero_progress_flag` is true only for a resolved progress of exactly 0.
- `long_running_flag` and `stalled_flag` stay `NULL`. Their `_rule_status` columns say what they wait for.
- `run_id` hashes the component run, taxonomy version, snapshot, and project rule version. The version moves to `silver_project_v3`, so dependent maps must be rerun after this notebook.

**Protects:** Completeness, Uniqueness, and Auditability.

**Expected result:** one row per Contract ID. The validator proves the counts.

In [0]:
%sql
CREATE OR REPLACE TABLE `02-silver`.silver_project
USING DELTA
COMMENT 'One canonical DPWH project per Contract ID per Silver run. Values publish only when components agree. Conflicts and quality findings stay visible.'
AS
SELECT
    attribute.contract_id,
    attribute.component_count,

    attribute.project_description_raw,
    attribute.project_description_raw_resolution_status AS description_resolution_status,
    attribute.distinct_project_description_raw_count AS distinct_description_count,

    attribute.category_raw,
    attribute.category_raw_resolution_status AS category_resolution_status,
    attribute.distinct_category_raw_count AS distinct_category_count,

    category.source_infra_types,
    category.source_infra_type,
    category.approved_standardized_sectors,
    category.approved_standardized_categories,
    category.unapproved_source_infra_types,
    category.standardized_sector,
    category.standardized_category,
    category.taxonomy_version,
    category.category_mapping_state,
    category.category_classification_status,
    category.is_dpwh_flood_related,

    attribute.status_raw,
    attribute.status_raw_resolution_status AS status_resolution_status,
    attribute.distinct_status_raw_count AS distinct_status_count,

    attribute.standardized_status,
    attribute.status_group,
    attribute.status_mapping_version,
    attribute.status_mapping_state,

    attribute.physical_progress_pct,
    attribute.physical_progress_pct_resolution_status AS progress_resolution_status,
    attribute.distinct_physical_progress_pct_count AS distinct_progress_count,
    attribute.progress_quality_status,

    attribute.start_date,
    attribute.start_date_resolution_status,
    attribute.distinct_start_date_count,

    attribute.completion_date,
    attribute.completion_date_resolution_status,
    attribute.distinct_completion_date_count,

    attribute.latitude_parsed,
    attribute.longitude_parsed,

    attribute.infra_year,
    attribute.infra_year_parsed,
    attribute.infra_year_parsed_resolution_status AS infra_year_resolution_status,

    attribute.reported_region,
    attribute.reported_region_resolution_status,
    attribute.distinct_reported_region_count,

    attribute.contractor,
    attribute.distinct_contractor_count,

    attribute.source_of_funds,
    attribute.distinct_source_of_funds_count AS distinct_funding_source_count,

    budget.reported_budget_pesos,
    budget.budget_resolution_status,
    budget.budget_quality_status,
    budget.distinct_budget_value_count,
    budget.valid_budget_component_count,

    CASE
        WHEN attribute.physical_progress_pct_resolution_status = 'RESOLVED'
            THEN attribute.physical_progress_pct = 0
    END AS zero_progress_flag,
    CAST(NULL AS BOOLEAN) AS stalled_flag,
    CAST(NULL AS BOOLEAN) AS long_running_flag,
    CASE
        WHEN attribute.physical_progress_pct_resolution_status = 'RESOLVED'
            THEN 'ZERO_PROGRESS_RULE_EVALUATED'
        ELSE 'UNRESOLVED_PROGRESS'
    END AS zero_progress_rule_status,
    'STATUS_MAPPING_REQUIRED' AS stalled_rule_status,
    'SOURCE_SNAPSHOT_DATE_RULE_REQUIRED' AS long_running_rule_status,

    attribute.source_system,
    attribute.source_snapshot_id,
    attribute.source_ingest_run_id,
    attribute.component_run_id,
    attribute.source_modified_at,

    SHA2(
        CONCAT_WS('|', parameters.project_key_label, COALESCE(attribute.contract_id, '')),
        256
    ) AS project_key,
    SHA2(
        CONCAT_WS(
            '||',
            COALESCE(attribute.source_snapshot_id, ''),
            COALESCE(attribute.component_run_id, ''),
            COALESCE(category.taxonomy_version, ''),
            parameters.transformation_rule_version
        ),
        256
    ) AS run_id,
    parameters.transformation_rule_version
FROM project_attribute_resolution AS attribute
LEFT JOIN project_budget_resolution AS budget
    ON attribute.contract_id = budget.contract_id
LEFT JOIN project_category_resolution AS category
    ON attribute.contract_id = category.contract_id
CROSS JOIN project_parameters AS parameters;

# Summary

- Created `02-silver.silver_project` at one row per Contract ID, keyed by `project_key`.
- Every attribute follows one resolution rule. Distinct counts and published values use the same expression.
- Budgets are never summed. Conflicts stay `CONFLICT` with a `NULL` budget.
- Category arrays are unioned across components, sorted, and deduplicated before classification. One project remains one row.
- `source_infra_type`, `standardized_sector`, `taxonomy_version`, `category_mapping_state`, and `is_dpwh_flood_related` are owned by Silver for Gold reuse.
- Known limits: missing or unapproved component categories remain unmapped. Status, long-running, and stalled rules wait on team decisions.
- Next: `04_validation/04_validation_silver_projects`. Then rerun Tables 13 and 15 and their validators, because `run_id` changed.

The query below shows resolution coverage. The denominator is `project_rows`.

In [0]:
%sql
SELECT
    COUNT(*) AS project_rows,
    SUM(component_count) AS component_rows,
    COUNT_IF(budget_resolution_status = 'CONFLICT') AS budget_conflicts,
    COUNT_IF(budget_resolution_status = 'MISSING_OR_INVALID') AS budget_missing,
    COUNT_IF(progress_quality_status = 'OUT_OF_RANGE') AS progress_out_of_range,
    COUNT_IF(reported_region_resolution_status <> 'RESOLVED') AS region_unresolved,
    COUNT_IF(category_mapping_state = 'MAPPED') AS category_mapped,
    COUNT_IF(category_mapping_state = 'UNMAPPED') AS category_unmapped,
    COUNT_IF(category_classification_status = 'MAPPED_SINGLE') AS single_sector_projects,
    COUNT_IF(category_classification_status = 'MAPPED_MULTI') AS multi_sector_projects,
    COUNT_IF(category_classification_status = 'MISSING_SOURCE_CATEGORY') AS missing_category_projects,
    COUNT_IF(category_classification_status = 'UNAPPROVED_SOURCE_CATEGORY') AS unapproved_category_projects,
    COUNT_IF(is_dpwh_flood_related) AS dpwh_flood_related_projects,
    SUM(CASE WHEN is_dpwh_flood_related THEN reported_budget_pesos ELSE CAST(0 AS DECIMAL(18, 2)) END) AS dpwh_flood_related_reported_budget_pesos,
    SUM(CASE WHEN category_mapping_state = 'MAPPED' THEN reported_budget_pesos ELSE CAST(0 AS DECIMAL(18, 2)) END) AS mapped_reported_budget_pesos,
    SUM(CASE WHEN category_mapping_state = 'UNMAPPED' THEN reported_budget_pesos ELSE CAST(0 AS DECIMAL(18, 2)) END) AS unmapped_reported_budget_pesos,
    COUNT_IF(status_mapping_state = 'MAPPED') AS status_mapped,
    SUM(reported_budget_pesos) AS resolved_reported_budget_pesos
FROM `02-silver`.silver_project;